In [37]:
%load_ext sql
%config SqlMagic.autocommit = True
%config SqlMagic.displaylimit = None
%sql duckdb://



The sql extension is already loaded. To reload it, use:
  %reload_ext sql


displaylimit: Value None will be treated as 0 (no limit)

In [29]:
crashes_crashes = "D:/PythonProjects/Chicago Traffic Crashes/export.csv"
crashes_veh = "D:/PythonProjects/Chicago Traffic Crashes/crashes_vehicles.csv"

In [43]:
%%sql
CREATE OR REPLACE VIEW crashes AS 
SELECT * FROM read_csv_auto('D:/PythonProjects/Chicago Traffic Crashes/crashes_crashes.csv');

CREATE OR REPLACE VIEW vehicles AS 
SELECT * FROM read_csv_auto('D:/PythonProjects/Chicago Traffic Crashes/crashes_vehicles.csv');

Running query in 'duckdb://'

Count


In [90]:
%%sql
CREATE OR REPLACE VIEW crashes_clean AS
SELECT * FROM crashes
WHERE CAST(crash_date AS TIMESTAMP) >= '2018-01-01'
  AND CAST(crash_date AS TIMESTAMP) <  '2026-01-01';

Running query in 'duckdb://'

Count


In [52]:
%%sql
CREATE OR REPLACE VIEW fleet_vehicles AS 
SELECT *,
    CASE
        WHEN vehicle_type = 'TRUCK - SINGLE UNIT' THEN 'local_truck'
        WHEN vehicle_type = 'VAN/MINI-VAN' AND vehicle_use LIKE 'COMMERCIAL%' THEN 'commercial_van'
        WHEN vehicle_type LIKE 'TRACTOR%' THEN 'freight'
    END AS fleet_group
FROM vehicles
WHERE fleet_group IS NOT NULL;


Running query in 'duckdb://'

Count


In [54]:
%%sql
SELECT fleet_group,
       COUNT(*) AS vehicle_rows,
       COUNT(DISTINCT crash_record_id) AS crashes
FROM fleet_vehicles
GROUP BY fleet_group
ORDER BY vehicle_rows DESC;

Running query in 'duckdb://'

fleet_group,vehicle_rows,crashes
local_truck,41123,39784
freight,23218,21893
commercial_van,4372,4307


In [55]:
%%sql
SELECT vehicle_use IS NULL AS use_missing, COUNT(*) AS n
FROM vehicles
WHERE vehicle_type = 'VAN/MINI-VAN'
GROUP BY ALL;

Running query in 'duckdb://'

use_missing,n
False,100103


In [91]:
%%sql
WITH grp AS (SELECT DISTINCT crash_record_id, fleet_group FROM fleet_vehicles)
SELECT g.fleet_group,
       COUNT(*) AS crashes,
       ROUND(100.0 * COUNT(*) FILTER (WHERE c.injuries_total > 0) / COUNT(*), 1) AS injury_pct
FROM grp g
JOIN crashes_clean c USING (crash_record_id)
GROUP BY g.fleet_group
UNION ALL
SELECT 'all_crashes', COUNT(*),
       ROUND(100.0 * COUNT(*) FILTER (WHERE injuries_total > 0) / COUNT(*), 1)
FROM crashes;

Running query in 'duckdb://'

fleet_group,crashes,injury_pct
local_truck,31789,8.9
freight,18026,8.8
commercial_van,3337,10.5
all_crashes,1110911,14.6


In [92]:
%%sql
COPY(
    WITH grp AS (SELECT DISTINCT crash_record_id, fleet_group FROM fleet_vehicles)
    SELECT
        EXTRACT(HOUR FROM CAST(c.crash_date AS TIMESTAMP)) AS hour_of_day,
        g.fleet_group,
        COUNT(*) AS crashes,
        ROUND(100.0 * COUNT(*) FILTER (WHERE c.injuries_total > 0) / COUNT(*), 1) AS injury_pct
    FROM grp g
    JOIN crashes_clean c USING (crash_record_id)
    WHERE c.crash_date IS NOT NULL
    GROUP BY ALL
    ORDER BY fleet_group, hour_of_day
) TO 'D:/PythonProjects/Chicago Traffic Crashes/Out/hourly_volume_severity.csv' (HEADER);

Running query in 'duckdb://'

Count


In [106]:
%%sql
COPY(
    WITH grp AS (SELECT DISTINCT crash_record_id, fleet_group FROM fleet_vehicles
                    WHERE fleet_group IN ('local_truck','freight'))
        SELECT EXTRACT(HOUR FROM CAST(c.crash_date AS TIMESTAMP)) AS hour_of_day,
            g.fleet_group,
            COUNT(*) AS crashes,
            COUNT(*) FILTER (WHERE c.injuries_total > 0) AS injury_crashes
        FROM grp g JOIN crashes_clean c USING (crash_record_id)
        GROUP BY ALL
        ORDER BY 2, 1
) TO 'D:/PythonProjects/Chicago Traffic Crashes/Out/hourly_volume_severity.csv' (HEADER);

Running query in 'duckdb://'

Count


In [93]:
%%sql
COPY(
     WITH grp AS (SELECT DISTINCT crash_record_id, fleet_group FROM fleet_vehicles)
     SELECT g.fleet_group,
          CASE WHEN c.crash_hour BETWEEN 0 AND 5  THEN '1 overnight (12-5am)'
               WHEN c.crash_hour BETWEEN 6 AND 9  THEN '2 morning rush (6-9am)'
               WHEN c.crash_hour BETWEEN 10 AND 15 THEN '3 midday (10am-3pm)'
               WHEN c.crash_hour BETWEEN 16 AND 19 THEN '4 evening rush (4-7pm)'
               ELSE '5 night (8-11pm)'
          END AS time_block,
          COUNT(*) AS crashes,
          ROUND(100.0 * COUNT(*) FILTER (WHERE c.injuries_total > 0) / COUNT(*), 1) AS injury_pct
     FROM grp g
     JOIN crashes_clean c USING (crash_record_id)
     GROUP BY ALL
     ORDER BY 1, 2
) TO 'D:/PythonProjects/Chicago Traffic Crashes/Out/injury_by_block.csv' (HEADER);

Running query in 'duckdb://'

Count


Lighting conditions

In [97]:
%%sql
COPY(
     WITH grp AS (SELECT DISTINCT crash_record_id, fleet_group FROM fleet_vehicles
               WHERE fleet_group IN ('local_truck','freight'))
     SELECT CASE WHEN EXTRACT(HOUR FROM CAST(c.crash_date AS TIMESTAMP)) BETWEEN 0 AND 5 THEN 'overnight'
               WHEN EXTRACT(HOUR FROM CAST(c.crash_date AS TIMESTAMP)) BETWEEN 10 AND 15 THEN 'midday'
          END AS time_block,
          c.lighting_condition,
          COUNT(*) AS crashes,
          ROUND(100.0 * COUNT(*) FILTER (WHERE c.injuries_total > 0) / COUNT(*), 1) AS injury_pct
     FROM grp g
     JOIN crashes_clean c USING (crash_record_id)
     WHERE time_block IS NOT NULL
     GROUP BY ALL
     HAVING COUNT(*) >= 100
     ORDER BY time_block, crashes DESC
) TO 'D:/PythonProjects/Chicago Traffic Crashes/Out/lighting.csv' (HEADER);

Running query in 'duckdb://'

Count


Crash type by time block

In [98]:
%%sql
COPY(
    WITH grp AS (SELECT DISTINCT crash_record_id FROM fleet_vehicles
                WHERE fleet_group IN ('local_truck','freight')),
    base AS (
        SELECT CASE WHEN EXTRACT(HOUR FROM CAST(c.crash_date AS TIMESTAMP)) BETWEEN 0 AND 5 THEN 'overnight'
                    WHEN EXTRACT(HOUR FROM CAST(c.crash_date AS TIMESTAMP)) BETWEEN 10 AND 15 THEN 'midday'
            END AS time_block,
            c.first_crash_type,
            c.injuries_total
        FROM grp g
        JOIN crashes_clean c USING (crash_record_id)
    )
    SELECT time_block,
           first_crash_type,
        COUNT(*) AS crashes,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY time_block), 1) AS pct_of_block,
        ROUND(100.0 * COUNT(*) FILTER (WHERE injuries_total > 0) / COUNT(*), 1) AS injury_pct
    FROM base
    WHERE time_block IS NOT NULL
    GROUP BY time_block, first_crash_type
    HAVING COUNT(*) >= 50
    ORDER BY time_block, crashes DESC
) TO 'D:/PythonProjects/Chicago Traffic Crashes/Out/crash_type.csv' (HEADER);

Running query in 'duckdb://'

Count


In [104]:
%%sql
WITH grp AS (SELECT DISTINCT crash_record_id FROM fleet_vehicles
                WHERE fleet_group IN ('local_truck','freight')),
    base AS (
        SELECT CASE WHEN EXTRACT(HOUR FROM CAST(c.crash_date AS TIMESTAMP)) BETWEEN 0 AND 5 THEN 'overnight'
                    WHEN EXTRACT(HOUR FROM CAST(c.crash_date AS TIMESTAMP)) BETWEEN 10 AND 15 THEN 'midday'
            END AS time_block,
            c.first_crash_type,
            c.injuries_total
        FROM grp g
        JOIN crashes_clean c USING (crash_record_id)
    )
    SELECT time_block,
           first_crash_type,
        COUNT(*) AS crashes,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY time_block), 1) AS pct_of_block,
        ROUND(100.0 * COUNT(*) FILTER (WHERE injuries_total > 0) / COUNT(*), 1) AS injury_pct
    FROM base
    WHERE time_block IS NOT NULL
    GROUP BY time_block, first_crash_type
    HAVING COUNT(*) >= 50
    ORDER BY time_block, crashes DESC;

Running query in 'duckdb://'

time_block,FIRST_CRASH_TYPE,crashes,pct_of_block,injury_pct
midday,PARKED MOTOR VEHICLE,7163,29.9,1.8
midday,SIDESWIPE SAME DIRECTION,4712,19.7,5.3
midday,TURNING,4059,16.9,12.0
midday,REAR END,3097,12.9,17.5
midday,FIXED OBJECT,2091,8.7,2.1
midday,ANGLE,976,4.1,21.3
midday,REAR TO FRONT,539,2.2,3.9
midday,SIDESWIPE OPPOSITE DIRECTION,437,1.8,7.6
midday,OTHER OBJECT,363,1.5,2.5
midday,PEDESTRIAN,157,0.7,82.8


In [107]:
%%sql
WITH grp AS (SELECT DISTINCT crash_record_id FROM fleet_vehicles
             WHERE fleet_group IN ('local_truck','freight')),
fleet AS (
    SELECT c.injuries_total,
           EXTRACT(HOUR FROM CAST(c.crash_date AS TIMESTAMP)) AS hr
    FROM grp g JOIN crashes_clean c USING (crash_record_id)
)
SELECT 'all_crashes' AS metric, COUNT(*) AS crashes,
       ROUND(100.0 * COUNT(*) FILTER (WHERE injuries_total > 0) / COUNT(*), 1) AS injury_pct
FROM crashes_clean
UNION ALL
SELECT 'truck_midday', COUNT(*),
       ROUND(100.0 * COUNT(*) FILTER (WHERE injuries_total > 0) / COUNT(*), 1)
FROM fleet WHERE hr BETWEEN 10 AND 15
UNION ALL
SELECT 'truck_overnight', COUNT(*),
       ROUND(100.0 * COUNT(*) FILTER (WHERE injuries_total > 0) / COUNT(*), 1)
FROM fleet WHERE hr BETWEEN 0 AND 5;

Running query in 'duckdb://'

metric,crashes,injury_pct
all_crashes,877919,15.1
truck_midday,24083,8.3
truck_overnight,2600,15.2


In [101]:
%%sql
SELECT EXTRACT(YEAR FROM CAST(crash_date AS TIMESTAMP)) AS yr,
       COUNT(*) AS crashes,
       ROUND(100.0 * COUNT(*) FILTER (WHERE injuries_total > 0) / COUNT(*), 1) AS injury_pct
FROM crashes_clean
GROUP BY yr
ORDER BY yr;

Running query in 'duckdb://'

yr,crashes,injury_pct
2018,118952,13.9
2019,117764,13.9
2020,92096,15.4
2021,108767,15.0
2022,108413,14.8
2023,110754,15.1
2024,112056,16.5
2025,109117,16.4


In [103]:
%%sql
WITH grp AS (SELECT DISTINCT crash_record_id FROM fleet_vehicles
             WHERE fleet_group IN ('local_truck','freight')),
base AS (
    SELECT EXTRACT(YEAR FROM CAST(c.crash_date AS TIMESTAMP)) AS yr,
           CASE WHEN EXTRACT(HOUR FROM CAST(c.crash_date AS TIMESTAMP)) BETWEEN 0 AND 5 THEN 'overnight'
                WHEN EXTRACT(HOUR FROM CAST(c.crash_date AS TIMESTAMP)) BETWEEN 10 AND 15 THEN 'midday'
           END AS time_block,
           c.injuries_total
    FROM grp g JOIN crashes_clean c USING (crash_record_id)
)
SELECT yr,
       COUNT(*) FILTER (WHERE time_block = 'overnight') AS overnight_crashes,
       ROUND(100.0 * COUNT(*) FILTER (WHERE time_block = 'overnight' AND injuries_total > 0)
             / COUNT(*) FILTER (WHERE time_block = 'overnight'), 1) AS overnight_injury_pct,
       COUNT(*) FILTER (WHERE time_block = 'midday') AS midday_crashes,
       ROUND(100.0 * COUNT(*) FILTER (WHERE time_block = 'midday' AND injuries_total > 0)
             / COUNT(*) FILTER (WHERE time_block = 'midday'), 1) AS midday_injury_pct
FROM base
GROUP BY yr
ORDER BY yr;

Running query in 'duckdb://'

yr,overnight_crashes,overnight_injury_pct,midday_crashes,midday_injury_pct
2018,364,12.9,3812,8.4
2019,325,15.7,3496,8.2
2020,249,11.6,2633,8.7
2021,334,12.9,3009,8.1
2022,354,17.2,2853,8.0
2023,307,16.6,2862,7.9
2024,326,19.3,2762,8.9
2025,341,15.0,2656,7.8


In [88]:
import pandas as pd

df = pd.read_csv(r"D:\PythonProjects\Chicago Traffic Crashes\crashes_crashes.csv", parse_dates=['CRASH_DATE'], date_format="%m/%d/%Y %I:%M:%S %p")
df2 = pd.read_csv(r"D:\PythonProjects\Chicago Traffic Crashes\crashes_vehicles.csv", parse_dates=['CRASH_DATE'], date_format="%m/%d/%Y %I:%M:%S %p")


print(df['CRASH_DATE'].max())
print(df2['CRASH_DATE'].max())

C:\Users\Juan\AppData\Local\Temp\ipykernel_38472\68646668.py:3: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r"D:\PythonProjects\Chicago Traffic Crashes\crashes_crashes.csv", parse_dates=['CRASH_DATE'], date_format="%m/%d/%Y %I:%M:%S %p")
C:\Users\Juan\AppData\Local\Temp\ipykernel_38472\68646668.py:4: DtypeWarning: Columns (3,18,20,38,39,40,41,43,47,48,49,52,54,57,58,60,64,70) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv(r"D:\PythonProjects\Chicago Traffic Crashes\crashes_vehicles.csv", parse_dates=['CRASH_DATE'], date_format="%m/%d/%Y %I:%M:%S %p")


2026-11-06 00:01:00
2026-11-06 00:01:00
